In [ ]:
# 4. RESULTS & VIZ — par typologie (5 cibles)
from pathlib import Path
import joblib
import pandas as pd
import numpy as np
import plotly.express as px
from sklearn.metrics import mean_squared_error

ROOT = Path("..").resolve()
DATA = ROOT / "data"
MODELS = ROOT / "models"
DOCS = ROOT / "docs"
VISUALS = ROOT / "visuals"
VISUALS.mkdir(exist_ok=True, parents=True)

DFP = DATA / "df_dummies_2019.csv"
sample = DFP.read_text(encoding="utf-8", errors="ignore")[:400]
df = pd.read_csv(DFP, sep=";" if sample.count(";") > sample.count(",") else ",")

registry = pd.read_json(MODELS / "registry.json")
metrics = pd.read_csv(DOCS / "model_metrics.csv")
display(metrics)

In [ ]:
def rmse(y_true, y_pred) -> float:
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

In [ ]:
# 1) Comparatif R²
fig = px.bar(
    metrics,
    x="target",
    y="R2_test",
    text=metrics["R2_test"].round(2),
    title="Performance des modèles (R² test) par typologie",
)
fig.update_traces(textposition="outside")
fig.show()

In [ ]:
# 2) Réel vs Prédit + 3) Résidus + 4) Importances (coefficients Lasso)
for target in registry["target"]:
    blob = joblib.load(MODELS / f"{target}.pkl")
    model, feats = blob["model"], blob["features"]

    X = df[feats]
    y_true = pd.to_numeric(df[target], errors="coerce")
    y_pred = model.predict(X)

    # Scatter Réel vs Prédit
    fig = px.scatter(
        x=y_true,
        y=y_pred,
        labels={"x": "Réel", "y": "Prédit"},
        title=f"Réel vs Prédit — {target}",
    )
    fig.add_shape(
        type="line",
        x0=y_true.min(),
        y0=y_true.min(),
        x1=y_true.max(),
        y1=y_true.max(),
        line=dict(color="red", dash="dash"),
    )
    fig.show()

    # Résidus
    resid = y_true - y_pred
    fig = px.histogram(
        resid,
        nbins=50,
        title=f"Résidus — {target}",
        labels={"value": "Erreur (réel - prédit)"},
    )
    fig.show()

    # Importances (coeffs)
    coefs = model.named_steps["lasso"].coef_
    coef_df = pd.DataFrame({"feature": feats, "coef": coefs})
    coef_df["abs_coef"] = coef_df["coef"].abs()
    top = coef_df.sort_values("abs_coef", ascending=False).head(15)
    fig = px.bar(
        top, x="coef", y="feature", orientation="h", title=f"Top |coef| — {target}"
    )
    fig.show()

In [ ]:
# 5) (option) Évolution temporelle réelle vs prédite si 'année/annee' est dispo dans df_dummies_2019.csv
COL_YEAR = (
    "année" if "année" in df.columns else ("annee" if "annee" in df.columns else None)
)
if COL_YEAR:
    for target in registry["target"]:
        blob = joblib.load(MODELS / f"{target}.pkl")
        model, feats = blob["model"], blob["features"]
        tmp = df[[COL_YEAR, target] + feats].copy()
        tmp["pred"] = model.predict(tmp[feats])
        trend = tmp.groupby(COL_YEAR)[[target, "pred"]].mean().reset_index()
        fig = px.line(
            trend,
            x=COL_YEAR,
            y=[target, "pred"],
            markers=True,
            title=f"Évolution moyenne annuelle — {target}",
        )
        fig.show()